In [1]:
import os
import tifffile
import rasterio

import cv2
import numpy as np

import leafmap.leafmap as leafmap
#from samgeo import SamGeo2

import geopandas as gpd
import pickle
from pyproj import Transformer

from utils.raster_tools import Raster_profile 
import matplotlib.pyplot as plt

### See the overall

In [2]:
clipped_theos_file = "theos/clipped_IMG_T2V_20250119034323_ORTHO_PMS_32_small.tif"

# m = leafmap.Map(center=[(lat_start + lat_end)/2, (long_start + long_end)/2], zoom=16, height="800px")
m = leafmap.Map(center=[12.908807, 100.922147], zoom=16, height="800px")
m.add_basemap("Satellite") 
m.add_raster(clipped_theos_file, layer_name="Theos") 
m

Map(center=[12.9264665, 100.90246099999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

## Read the cliped Google image

In [3]:
theos_Profile = Raster_profile(clipped_theos_file) 

Dataset name: theos/clipped_IMG_T2V_20250119034323_ORTHO_PMS_32_small.tif
File mode: r
Number of bands: 4
Image width: 7001 pixels
Image height: 7053 pixels
Coordinate Reference System (CRS): EPSG:32647
Data shape: (4, 7053, 7001)
Data type: uint16


### Specify the grid's spacing

In [4]:
from pyproj import Transformer
import csv

def setup_polygon(long_start, lat_start, long_end, lat_end, crs_source="EPSG:4326", crs_target="EPSG:32647"): 
    bbox = [long_start, lat_start, long_end, lat_end] 
    coordinates = [
        [bbox[0], bbox[3]],  # Top-left corner (min_lon, max_lat)
        [bbox[2], bbox[3]],  # Top-right corner (max_lon, max_lat)
        [bbox[2], bbox[1]],  # Bottom-right corner (max_lon, min_lat)
        [bbox[0], bbox[1]],  # Bottom-left corner (min_lon, min_lat)
        [bbox[0], bbox[3]]   # Closing the polygon by repeating the first point
        ]

    transformer = Transformer.from_crs(crs_source, crs_target, always_xy=True)
    poly_gons = []
    for coord in coordinates: 
        easting, northing = transformer.transform(coord[0], coord[1])
        poly_gons.append([easting, northing])

    return poly_gons, bbox, coordinates

def save_stats(stats, path_npz, path_csv):
    np.savez(path_npz, **stats)

    with open(path_csv, 'w', newline='') as file:
        writer = csv.writer(file)
        # Write header
        writer.writerow(['Key', 'Value'])
        # Write data line by line
        for key, value in stats.items():
            writer.writerow([key, value])
            
def read_npz(npz_filename):
    read_stats = dict(np.load(npz_filename))

    read_dict = {}
    for key, value in read_stats.items():
        try:
            read_dict[key] = value.item() 
        except:
            read_dict[key] = value

    return read_dict

In [16]:
long_start, lat_start = theos_Profile.get_longlat_from_image_pixels(0, 0, crs_dst="EPSG:4326")

In [17]:
long_start

100.88645527094697

In [18]:
lat_start

12.942517599688498

In [6]:
long_end, lat_end = theos_Profile.get_longlat_from_image_pixels(7000, 7000, crs_dst="EPSG:4326")

In [7]:
lat_end

12.910650418615315

In [8]:
long_end

100.9184638035386

In [9]:
long_start_temp = long_start   
long_end_temp   = long_end  

lat_start_temp  = lat_start  
lat_end_temp    = lat_end 

# lat_start_temp  = lat_start  - (slice_no)*lat_diff.item()
#lat_end_temp  = lat_start  - (slice_no+1)*lat_diff.item()
crs_source = "EPSG:4326" # Google 
crs_target = "EPSG:32647" # Theos
 
print("Start: LON: %f LAT: %f" % (long_start_temp, lat_start_temp)) 
print("End  : LON: %f LAT: %f" % (long_end_temp, lat_end_temp)) 

poly_gons, bbox, coordinates = setup_polygon(long_start_temp, lat_start_temp, long_end_temp, lat_end_temp, crs_source=crs_source, crs_target=crs_target)

smaller_clipped_theos_file = "theos/clipped_IMG_T2V_20250119034323_ORTHO_PMS_32_small.tif"
leafmap.clip_image(clipped_theos_file, poly_gons, smaller_clipped_theos_file)

Start: LON: 100.886455 LAT: 12.942518
End  : LON: 100.918464 LAT: 12.910650


Reading input: c:\Users\user\Documents\Gistda_workspace\geo\Building_detection\theos\clipped_IMG_T2V_20250119034323_ORTHO_PMS_32_small.tif

Adding overviews...
Updating dataset tags...
Writing output to: c:\Users\user\Documents\Gistda_workspace\geo\Building_detection\theos\clipped_IMG_T2V_20250119034323_ORTHO_PMS_32_small.tif


In [10]:
# m = leafmap.Map(center=[(lat_start + lat_end)/2, (long_start + long_end)/2], zoom=16, height="800px")
m = leafmap.Map(center=[12.908807, 100.922147], zoom=16, height="800px")
m.add_basemap("Satellite") 
m.add_raster(clipped_theos_file, layer_name="Theos") 
m.add_raster(smaller_clipped_theos_file, layer_name="Theos") 
m

Map(center=[12.9264665, 100.90246099999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

In [11]:
long_start, lat_start = theos_Profile.get_longlat_from_image_pixels(0, 0, crs_dst="EPSG:4326")

In [12]:
zoom_level = 18 # Google resolution // the higher zoom level >> higher resolution 
long_diff  = 2*np.abs(100.9261026816609 - 100.9230026816609)
lat_diff   = 2*np.abs(13.054297390119896 - 13.052397390099848)
crs_source = "EPSG:4326" # Google 
crs_target = "EPSG:32647" # Theos

In [19]:
long_diff

np.float64(0.006199999999978445)

In [20]:
lat_diff

np.float64(0.003800000040094176)

In [13]:
slice_no   = 1

slices_path = "ISP0704-Zoom%d" % zoom_level
slice_subpath  = os.path.join(slices_path, "%0000d" % slice_no)
slice_google_filename = os.path.join(slices_path, "%0000d" % slice_no, "google.tif") 
warped_slice_google_filename = os.path.join(slices_path, "%0000d" % slice_no, "warped_google.tif") 
slice_theos_filename  = os.path.join(slices_path, "%0000d" % slice_no, "theos.tif") 
npz_filename  = os.path.join(slices_path, "%0000d" % slice_no, "stats.npz") 
csv_filename = os.path.join(slices_path, "%0000d" % slice_no, "stats.csv") 

os.makedirs(slices_path, exist_ok=True)
os.makedirs(slice_subpath, exist_ok=True)

long_start_temp = long_start  + (slice_no)*long_diff.item()   
long_end_temp = long_start + (slice_no+1)*long_diff.item()  

lat_start_temp  = lat_start  
lat_end_temp    = lat_start  - 1*lat_diff.item()

# lat_start_temp  = lat_start  - (slice_no)*lat_diff.item()
#lat_end_temp  = lat_start  - (slice_no+1)*lat_diff.item()

print("Slice no. %0000d" % slice_no) 
print("Start: LON: %f LAT: %f" % (long_start_temp, lat_start_temp)) 
print("End  : LON: %f LAT: %f" % (long_end_temp, lat_end_temp)) 

poly_gons, bbox, coordinates = setup_polygon(long_start_temp, lat_start_temp, long_end_temp, lat_end_temp, crs_source=crs_source, crs_target=crs_target)

leafmap.map_tiles_to_geotiff(output=slice_google_filename, bbox=bbox, zoom=zoom_level, source="Satellite", overwrite=True)
leafmap.clip_image(clipped_theos_file, poly_gons, slice_theos_filename)


stats = {"slice_no": slice_no, 
        "slice_theos_filename": slice_theos_filename,
        "slice_google_filename": slice_google_filename,  
        "long_start_temp": long_start_temp,
        "lat_start_temp": lat_start_temp,
        "long_end_temp": long_end_temp,
        "lat_end_temp": lat_end_temp,
        "poly_gons": poly_gons,
        "bbox":bbox,
        "coordinates": coordinates,
        "crs_source": crs_source,
        "crs_target": crs_target,
        "long_diff": long_diff,
        "lat_diff": lat_diff}

save_stats(stats, npz_filename, csv_filename)

Slice no. 1
Start: LON: 100.892655 LAT: 12.942518
End  : LON: 100.898855 LAT: 12.938718
Downloaded image 1/24
Downloaded image 2/24
Downloaded image 3/24
Downloaded image 4/24
Downloaded image 5/24
Downloaded image 6/24
Downloaded image 7/24
Downloaded image 8/24
Downloaded image 9/24
Downloaded image 10/24
Downloaded image 11/24
Downloaded image 12/24
Downloaded image 13/24
Downloaded image 14/24
Downloaded image 15/24
Downloaded image 16/24
Downloaded image 17/24
Downloaded image 18/24
Downloaded image 19/24
Downloaded image 20/24
Downloaded image 21/24
Downloaded image 22/24
Downloaded image 23/24
Downloaded image 24/24
Saving GeoTIFF. Please wait...
Image saved to ISP0704-Zoom18\1\google.tif


Reading input: c:\Users\user\Documents\Gistda_workspace\geo\Building_detection\ISP0704-Zoom18\1\theos.tif

Adding overviews...
Updating dataset tags...
Writing output to: c:\Users\user\Documents\Gistda_workspace\geo\Building_detection\ISP0704-Zoom18\1\theos.tif


## Test sliced data

### Stats reading

In [14]:
read_dict = read_npz(npz_filename)
read_dict

{'slice_no': 1,
 'slice_theos_filename': 'ISP0704-Zoom18\\1\\theos.tif',
 'slice_google_filename': 'ISP0704-Zoom18\\1\\google.tif',
 'long_start_temp': 100.89265527094695,
 'lat_start_temp': 12.942517599688498,
 'long_end_temp': 100.89885527094692,
 'lat_end_temp': 12.938717599648404,
 'poly_gons': array([[ 705326.18349865, 1431118.28804905],
        [ 705999.01809309, 1431123.27460236],
        [ 705995.89659046, 1431543.7094933 ],
        [ 705323.07220578, 1431538.72157616],
        [ 705326.18349865, 1431118.28804905]]),
 'bbox': array([100.89265527,  12.9425176 , 100.89885527,  12.9387176 ]),
 'coordinates': array([[100.89265527,  12.9387176 ],
        [100.89885527,  12.9387176 ],
        [100.89885527,  12.9425176 ],
        [100.89265527,  12.9425176 ],
        [100.89265527,  12.9387176 ]]),
 'crs_source': 'EPSG:4326',
 'crs_target': 'EPSG:32647',
 'long_diff': 0.006199999999978445,
 'lat_diff': 0.003800000040094176}

### Map reading

In [15]:
m = leafmap.Map()    
slice_google_filename_prev = 'ISP0704-Zoom18/0/google.tif'
slice_theos_filename_prev = 'ISP0704-Zoom18/0/theos.tif'
m.add_raster(slice_google_filename_prev, layer_name="Google-prev")
m.add_raster(slice_theos_filename_prev, layer_name="Theos-prev") 
m.add_raster(slice_google_filename, layer_name="Google")
m.add_raster(slice_theos_filename, layer_name="Theos") 
m

Map(center=[12.9405725, 100.895757], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title…

# Warping

In [ ]:
from plantcv import plantcv as pcv
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import tifffile
import os
from utils.tools import get_raster_data, image_enhancement, save_raster_and_write_meta

In [ ]:
slice_no   = 1

slices_path     = "ISP0704-Zoom%d" % zoom_level
slice_subpath   = os.path.join(slices_path, "%0000d" % slice_no)

slice_google_filename = os.path.join(slices_path, "%0000d" % slice_no, "google.tif") 
warped_slice_google_filename = os.path.join(slices_path, "%0000d" % slice_no, "warped_google.tif") 
slice_theos_filename  = os.path.join(slices_path, "%0000d" % slice_no, "theos.tif")  
path_warp_npz  = os.path.join(slices_path, "%0000d" % slice_no, "warp_stats.npz")  
path_warp_csv  = os.path.join(slices_path, "%0000d" % slice_no, "warp_stats.csv")   


In [ ]:
imgA, _ = get_raster_data(slice_google_filename) 
imgA = np.transpose(imgA, (1, 2, 0))  # Convert from (bands, height, width) to (height, width, bands)
 
imgB, _  = get_raster_data(slice_theos_filename) 
imgB = np.transpose(imgB, (1, 2, 0))  # Convert from (bands, height, width) to (height, width, bands)
imgB = imgB[:,:,:3] # 
imgB = image_enhancement(imgB)

In [ ]:
from utils.interactive_tools import Find_correspondences

%matplotlib widget
marker_AB = Find_correspondences(imgA, imgB, figsize=(10, 5))

In [ ]:
import cv2

point_src  = np.array(marker_AB.points[0])
point_dst  = np.array(marker_AB.points[1])

Homography, status = cv2.findHomography(point_src, point_dst) 

In [ ]:
target_size = (imgB.shape[1], imgB.shape[0])
im_dst = cv2.warpPerspective(imgA, Homography, target_size) 

In [ ]:
warp_stats = {
    "point_src": point_src,
    "point_dst": point_dst,
    "Homography": Homography,
    "target_size": target_size, 
    "src_img_filename": slice_google_filename,
    "dst_img_filename": slice_theos_filename,
    "result_image_filename": warped_slice_google_filename
}

In [ ]:
save_stats(warp_stats, path_warp_npz, path_warp_csv)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10,5)) 
axs[0].imshow(imgA[:,:,0:3])  # Display the first three channels (RGB) of the clipped image
axs[0].set_title('Google') # Set a title for the first subplot 

axs[1].imshow(imgB[:,:,0:3])  # Display the first three channels (RGB) of the clipped satellite image
axs[1].set_title('Theos') # Set a title for the second subplot 

axs[2].imshow(im_dst[:,:,0:3])  # Display the first three channels (RGB) of the clipped satellite image
axs[2].set_title('Google (warped)') # Set a title for the second subplot 
 
fig.tight_layout()

In [ ]:
im_dst_4D = np.zeros((imgB.shape[0], imgB.shape[1], 4)) 
im_dst_4D[:,:,:3] = im_dst 
im_dst_4D[:,:, 3] = 254
im_dst_4D = im_dst_4D.transpose(2, 0, 1)
im_dst_4D = im_dst_4D.astype(np.uint8)

destination_tif = warped_slice_google_filename
meta_source_tif = slice_theos_filename
save_raster_and_write_meta(im_dst_4D , destination_tif, meta_source_tif)

In [ ]:
import leafmap.leafmap as leafmap
m = leafmap.Map()
m.add_raster(slice_google_filename, layer_name="Google") 
m.add_raster(slice_theos_filename, layer_name="theos") 
m.add_raster(warped_slice_google_filename, layer_name="Google (warped)")  
m

## SamGeo2

In [ ]:
from samgeo import SamGeo2 

In [ ]:
sam = SamGeo2(
    model_id="sam2-hiera-large", 
    automatic=False
)

In [ ]:
# slice_no   = 0

slices_path     = "ISP0704-Zoom%d" % zoom_level
slice_subpath   = os.path.join(slices_path, "%0000d" % slice_no) 
warped_slice_google_filename = os.path.join(slices_path, "%0000d" % slice_no, "warped_google.tif")  
target_dir      = os.path.join(slices_path, "%0000d" % slice_no, "samgeo2mask")  
os.makedirs(target_dir, exist_ok=True)


In [ ]:
print("PLEASE SAVE THE MASK under Folder: %s" % target_dir)

In [ ]:
sam.set_image(warped_slice_google_filename)

In [ ]:
sam.show_map()

In [ ]:
mask_filename = os.path.join(slices_path, "%0000d" % slice_no, "samgeo2mask", "masks.tif") 

In [ ]:
import leafmap.leafmap as leafmap
m = leafmap.Map() 
m.add_raster(slice_theos_filename, layer_name="theos") 
m.add_raster(warped_slice_google_filename, layer_name="Google (warped)")  
m.add_raster(mask_filename, cmap="jet", layer_name="Mask (warped)")  
m

## Mask improvement

In [ ]:
from utils.mask_tools import Mask_profile

from plantcv import plantcv as pcv
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import tifffile

from utils.tools import get_raster_data, image_enhancement, save_raster_and_write_meta

In [ ]:
slice_no =0
mask_filename = os.path.join(slices_path, "%0000d" % slice_no, "samgeo2mask", "masks.tif") 
center_geojson = os.path.join(slices_path, "%0000d" % slice_no, "samgeo2mask", "masks_fg_markers.geojson")  
warped_slice_google_filename = os.path.join(slices_path, "%0000d" % slice_no, "warped_google.tif") 
sat_image, _ = get_raster_data(warped_slice_google_filename)

In [ ]:
Mask_obj = Mask_profile(mask_filename, center_geojson_file=center_geojson, center_geojson_crs="EPSG:4326") 
Mask_obj.show_mask_order(figsize=(10, 5), fontsize=11, alpha=0.75, satellite_image=sat_image)

In [ ]:
sat_image.shape
sat_image = sat_image.transpose(1,2,0)[:,:,:3]

In [ ]:
gray_bf_operation = Mask_obj.mask.copy()

for manual_index in range(Mask_obj.mask.max()):

    try: 
        gray_bf = Mask_obj.get_a_binary_mask(manual_index)

        record_af_list = []
        gray_af, record_af = Mask_obj.filling_holes(1*gray_bf)
        record_af_list.append(record_af)
        gray_af, record_af = Mask_obj.erosion(gray_af.astype(np.uint8), kernel_size=5)
        record_af_list.append(record_af)
        gray_af, record_af = Mask_obj.dilation(gray_af.astype(np.uint8), kernel_size=17)
        record_af_list.append(record_af)

        mask_2D = Mask_obj.update_a_binary_mask(gray_af, manual_index, record_af_list) 

    except:
        continue

    # fig, axs = plt.subplots(1, 2, figsize=(10, 3))

    # axs[0].imshow(sat_image)
    # axs[0].imshow(gray_bf, cmap='Oranges', alpha=0.45)
    # axs[0].set_title("Mask id %d before [top]" % manual_index)
    

    # axs[1].imshow(sat_image)
    # axs[1].imshow(gray_af, cmap='Oranges', alpha=0.45)
    # axs[1].set_title("Mask id %d after [top]" % manual_index) 

    # plt.tight_layout() 

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(sat_image)
axs[0].imshow(gray_bf_operation, cmap='turbo', alpha=0.35) 
axs[0].set_title("before")

axs[1].imshow(sat_image)
axs[1].imshow(mask_2D, cmap='turbo', alpha=0.35) 
axs[1].set_title("after")

In [ ]:
edited_mask_filename = os.path.join(slices_path, "%0000d" % slice_no, "samgeo2mask", "masks_edited.tif") 
meta_mask_filename   = os.path.join(slices_path, "%0000d" % slice_no, "samgeo2mask", "masks.tif") 
mask_2D              = mask_2D.reshape(1, mask_2D.shape[0], mask_2D.shape[1])
save_raster_and_write_meta(mask_2D, edited_mask_filename, meta_mask_filename)

In [ ]:
m = leafmap.Map()
m.add_raster(warped_slice_google_filename, layer_name="Image") 
m.add_circle_markers_from_xy(center_geojson, radius=3, color="red", fill_color="yellow", fill_opacity=0.8
) 
m.add_raster(mask_filename, cmap="jet", layer_name="Building masks (before)") 
m.add_raster(edited_mask_filename, cmap="jet", layer_name="Building masks (after)") 
m

In [ ]:
bb_filename = os.path.join(slices_path, "%0000d" % slice_no, "samgeo2mask", "boundbox.geojson") 
 
gdf = Mask_obj.make_boundboxes()
gdf.to_file(bb_filename, driver='GeoJSON') 

In [ ]:
m = leafmap.Map()
m.add_raster(warped_slice_google_filename, layer_name="Image") 
m.add_circle_markers_from_xy(center_geojson, radius=3, color="red", fill_color="yellow", fill_opacity=0.8
)  
m.add_raster(edited_mask_filename, cmap="jet", layer_name="Building masks (after)")  
m.add_vector(bb_filename, layer_name="Bounding Boxes")
m